# A3 — Koopman Feature-Space Geometry

**Goal:** test whether the fixed (untrained, `rff_trainable: false`) dynamics-embedding lift
shows a mechanistic signature — poor local linearizability (eDMD residual) and/or high
sensitivity amplification (Jacobian norm) — on chaotic ODE classes (Lorenz, Rossler, SprottB)
relative to the non-chaotic aperiodic class (Burgers, nu=1.0) and the periodic reference class
(Harmonic), that would explain the A1 behavioral pattern (ablation beats baseline on chaotic
in-distribution/held-out systems; baseline beats ablation on Burgers; ablation beats baseline
on Harmonic).

**This notebook only uses the `baseline_100k` checkpoint** (`use_dynamics_embedding=True`) —
the ablation checkpoint has no lift to analyze.

**Pre-registered interpretation map** (fill in before reading results, don't retrofit):

| Pre-projection (raw dictionary Φ) | Post-projection (392→512 learned) | Reading |
|---|---|---|
| bad on chaotic, good on Burgers | bad on chaotic, good on Burgers | Clean mechanistic confirmation of A1's behavioral split |
| bad on chaotic, good on Burgers | good everywhere | Lift is bad in principle but the trained downstream projection compensates — A1's split needs a *different* explanation (points toward A2a, temporal attention) |
| good everywhere | good everywhere | Negative result — lift geometry does not explain A1 at all → push to A2a |
| bad everywhere (no class separation) | — | Lift is uniformly poor; A1's *selectivity* is not explained by linear-fit quality alone |

**Design decisions locked in (see chat discussion):**
- Both pre-projection (Φ_pre, raw eDMD dictionary: raw patch + random polynomial feats + random
  Fourier feats, concatenated) and post-projection (Φ_post, the learned 392→512 linear map's
  output — what the transformer actually consumes) are tested, kept in **separate result
  tables**, never averaged.
- eDMD is fit **patch-to-patch** (Φ(P_t) → Φ(P_{t+1})), matching the granularity Panda's lift
  actually operates at — NOT the continuous-time Koopman operator.
- **Shared K** (one ridge-regularized linear operator fit across a class-balanced pool of
  trajectories), not per-class K — because the lift is architecturally fixed/global, so a
  per-class-optimal K would test something the model never has access to.
- Ridge regularization, λ chosen by cross-validation (not hand-picked).
- Normalized residual `||Φ(P_t+1) - K Φ(P_t)|| / ||Φ(P_t+1)||`, reported as median + IQR,
  Wilcoxon signed-rank test between class pairs on held-out trajectories.
- **3 random fit/held-out splits** for eDMD (robustness check, motivated by this project's
  earlier n=8-vs-n=20 heterogeneity scare).
- Jacobian sensitivity: single pass, no CV needed (it's a direct measurement, not a fit).

**Budget:** ~40 fit + 20 held-out trajectories/class x 3 splits for eDMD (feature extraction is
forward-pass-only, cheap; eDMD fitting itself is CPU/numpy, no GPU). ~15 trajectories/class,
single pass, for Jacobian sensitivity (backward-pass, the expensive part). Est. 2-3 GPU-hours
total out of this week's 20-hour Kaggle quota.

**Known limitations, stated up front (do not discover these later and treat as new news):**
- This tests whether *this specific pattern* (from A1's 5 tested classes) has a geometric
  correlate — it does not validate the periodic/aperiodic framing as a general theory, since no
  new systems are introduced.
- Lorenz/Rossler/SprottB vs. Burgers/Harmonic differ on multiple axes at once (chaoticity,
  channel count, ODE-vs-PDE origin) — a positive result here cannot by itself say *which* axis
  drives the effect.
- N=20 held-out/class supports directional/significance claims, not precise effect-size
  estimates.


## Section 1 — Setup

In [ ]:

import os, sys, json, glob
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}')

# ADJUST if your Kaggle session clones panda elsewhere
PANDA_REPO_PATH = '/kaggle/working/panda'
if PANDA_REPO_PATH not in sys.path:
    sys.path.insert(0, PANDA_REPO_PATH)

from panda.patchtst.pipeline import PatchTSTPipeline

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)


## Section 2 — Locate and load `baseline_100k` checkpoint

Robust locator (search-by-content, not hardcoded path depth) — same pattern used throughout
this project after being bitten twice by nesting mismatches. Searches for a directory whose
`config.json` has `use_dynamics_embedding: true`, under a dataset-root hint you set below.

In [ ]:

# ADJUST to your actual Kaggle dataset mount point
DATASET_ROOT_HINT = '/kaggle/input/datasets/anujb2'

def find_checkpoint_by_flag(root, want_use_dynamics_embedding=True):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if 'config.json' in filenames:
            try:
                with open(os.path.join(dirpath, 'config.json')) as f:
                    cfg = json.load(f)
            except Exception:
                continue
            if cfg.get('use_dynamics_embedding') == want_use_dynamics_embedding:
                # must actually have weights
                if any(fn.startswith('model.safetensors') or fn.startswith('pytorch_model')
                       for fn in filenames):
                    matches.append(dirpath)
    return matches

candidates = find_checkpoint_by_flag(DATASET_ROOT_HINT, want_use_dynamics_embedding=True)
print(f'Found {len(candidates)} candidate baseline (use_dynamics_embedding=True) checkpoint dir(s):')
for c in candidates:
    print(' ', c)

assert len(candidates) >= 1, (
    'No matching checkpoint found -- adjust DATASET_ROOT_HINT above, or confirm the checkpoint '
    'was uploaded with an intact config.json.'
)
if len(candidates) > 1:
    print('\n[WARNING] multiple matches -- pick the 100k one explicitly below rather than '
          'trusting candidates[0].')

# ADJUST: pick the correct index / path once you've inspected the printed list above
BASELINE_100K_DIR = candidates[0]
print(f'\nUsing: {BASELINE_100K_DIR}')

pipe = PatchTSTPipeline.from_pretrained(
    mode='predict', pretrain_path=BASELINE_100K_DIR, device_map=device,
)
model = pipe.model
model.eval()
print('Loaded baseline_100k pipeline.')


## Section 3 — Locate the dynamics-embedding module and register hooks

We don't hardcode the module path (repo internals may differ from memory) — instead we search
`model.named_modules()` for the lift by class-name / attribute signature, print candidates, and
let you confirm before hooking. The lift's `forward` should take a patch tensor and return the
concatenated [raw | poly | rff] dictionary (Φ_pre) as an intermediate, then a learned linear
layer projects it to d_model (Φ_post). If the module is a single fused block that only exposes
the *final* projected output, we capture the hook's **input** as Φ_pre and **output** as Φ_post
(the input to a `nn.Linear` projection layer, if present, is the pre-projection dictionary).

In [ ]:

print('Modules with "dyn" or "embed" or "koopman" or "lift" in their name:\n')
candidate_modules = []
for name, mod in model.named_modules():
    lname = name.lower()
    cname = mod.__class__.__name__.lower()
    if any(k in lname or k in cname for k in ['dynamics', 'koopman', 'lift', 'rff', 'poly', 'dict']):
        candidate_modules.append((name, mod.__class__.__name__))
        print(f'  {name:50s}  {mod.__class__.__name__}')

assert candidate_modules, (
    'No lift-like module found by name search. Print model to inspect manually:\n'
    '  print(model)\n'
    'then set DYNAMICS_EMBED_MODULE_NAME by hand below.'
)


In [ ]:

# CONFIRM: set this to the module name you want to hook, from the printed list above.
# This should be the dynamics-embedding / lift module itself (not a parent container).
DYNAMICS_EMBED_MODULE_NAME = candidate_modules[0][0]
print(f'Hooking module: {DYNAMICS_EMBED_MODULE_NAME}')

_captured = {'pre': None, 'post': None}

def _hook_fn(module, inputs, output):
    # inputs[0]: what goes IN to the lift module (the raw patch, shape [..., patch_len])
    # output: what the lift module itself returns.
    # We treat the module's OWN internal pre-projection dictionary as unavailable unless
    # exposed; as a robust default we capture:
    #   Φ_pre  = module's raw dictionary if the module exposes `.last_dict` (set this
    #            attribute inside the repo's forward if available), else we fall back to
    #            treating the hooked module's output as Φ_post only, and hook one level
    #            deeper (the pre-projection Linear layer) separately below.
    _captured['post'] = output.detach()
    if hasattr(module, 'last_dict'):
        _captured['pre'] = module.last_dict.detach()

target_module = dict(model.named_modules())[DYNAMICS_EMBED_MODULE_NAME]
handle_post = target_module.register_forward_hook(_hook_fn)

# Try to additionally find an inner nn.Linear (the 392->512 projection) to hook its INPUT
# as Φ_pre directly -- more robust than relying on a `.last_dict` attribute existing.
inner_linear_name, inner_linear = None, None
for name, mod in target_module.named_modules():
    if isinstance(mod, torch.nn.Linear):
        inner_linear_name, inner_linear = name, mod
        break  # first Linear inside the lift is almost certainly the dict->d_model projection

def _pre_hook_fn(module, inputs):
    _captured['pre'] = inputs[0].detach()

handle_pre = None
if inner_linear is not None:
    handle_pre = inner_linear.register_forward_pre_hook(_pre_hook_fn)
    print(f'Found inner projection Linear at "{DYNAMICS_EMBED_MODULE_NAME}.{inner_linear_name}" '
          f'(in_features={inner_linear.in_features}, out_features={inner_linear.out_features}) '
          '-- hooking its input as Phi_pre.')
else:
    print('[WARNING] No inner nn.Linear found inside the lift module. Phi_pre will only be '
          'populated if the module sets a `.last_dict` attribute during forward(). Inspect '
          'the module source before trusting Phi_pre results:')
    print(target_module)


## Section 4 — Sanity check the hooks with one dummy forward pass

Confirm shapes before running the full extraction loop. **Do not proceed past this cell until
both Φ_pre and Φ_post are populated with sensible shapes** (Φ_pre roughly `[..., 392]` per the
architecture notes: 16 raw + 120 poly + 256 rff for the documented config; adjust expectation
if `num_poly_feats`/`num_rff`/patch length differ in `model.config`; Φ_post should be
`[..., d_model]`).

In [ ]:

cfg = model.config
print('Relevant config fields:')
for k in ['use_dynamics_embedding', 'num_poly_feats', 'poly_degrees', 'num_rff',
          'rff_trainable', 'rff_scale', 'patch_length', 'd_model', 'context_length']:
    if hasattr(cfg, k):
        print(f'  {k} = {getattr(cfg, k)}')

patch_length = getattr(cfg, 'patch_length', 16)
context_length = getattr(cfg, 'context_length', 512)
n_channels_dummy = 3

dummy = torch.randn(1, n_channels_dummy, context_length, device=device)
with torch.no_grad():
    _ = pipe.model(dummy) if hasattr(pipe, 'model') else model(dummy)

print('\nPhi_pre  shape:', None if _captured['pre'] is None else tuple(_captured['pre'].shape))
print('Phi_post shape:', None if _captured['post'] is None else tuple(_captured['post'].shape))

assert _captured['post'] is not None, 'Hook did not fire -- check DYNAMICS_EMBED_MODULE_NAME and model forward signature (may need pipe.predict(...) instead of a raw model(dummy) call).'
if _captured['pre'] is None:
    print('\n[WARNING] Phi_pre not captured. eDMD/Jacobian analysis will be POST-PROJECTION ONLY '
          'until this is fixed -- go inspect the lift module source (`print(target_module)` above, '
          'or view the repo file directly) and adjust the hook.')


## Section 5 — Trajectory generators (5 classes)

Reuses this project's established generators verbatim where already defined elsewhere
(Burgers: `T=1500, N_x=128, nu=1.0`, PCA to 16 channels, matching Experiment 10/28's protocol).
Lorenz/Rossler/SprottB use the `dysts` library (as used throughout this project's held-out
system evaluation). Harmonic oscillator is a simple closed-form ODE.

**CONFIRM:** paste in the exact `load_burgers_nu1`, Lorenz/Rossler/SprottB trajectory
generators, and `load_harmonic` from `panda_100k_eval_clean.ipynb` / `new_experiments.ipynb`
here rather than reimplementing from scratch — those are already verified against this
project's conventions (fixed IC / RK4 / augmentation choices matter for consistency with A1).
Placeholder stubs below raise `NotImplementedError` until filled in, so this notebook fails
loudly instead of silently running on a different data distribution than A1 used.

In [ ]:

def load_lorenz_trajectories(n_traj, length=2048, seed=0):
    \"\"\"CONFIRM: paste verbatim from panda_100k_eval_clean.ipynb (the Lorenz gate_3ch
    protocol generator) -- must match A1's exact simulator (fixed IC / RK4 / 3-channel) so
    A3's classes correspond to what A1 actually tested.\"\"\"
    raise NotImplementedError('Paste verbatim Lorenz generator from panda_100k_eval_clean.ipynb')

def load_rossler_trajectories(n_traj, length=2048, seed=0):
    \"\"\"CONFIRM: paste verbatim from the held_out_trajectories cell used in A1.\"\"\"
    raise NotImplementedError('Paste verbatim Rossler generator from A1 held-out cell')

def load_sprottb_trajectories(n_traj, length=2048, seed=0):
    \"\"\"CONFIRM: paste verbatim from the held_out_trajectories cell used in A1.\"\"\"
    raise NotImplementedError('Paste verbatim SprottB generator from A1 held-out cell')

def load_burgers_nu1_trajectories(n_traj, seed=0):
    \"\"\"CONFIRM: paste verbatim -- T=1500, N_x=128, nu=1.0, PCA to 16 channels
    (Experiment 10/28 protocol).\"\"\"
    raise NotImplementedError('Paste verbatim Burgers nu=1.0 generator (T=1500, N_x=128, PCA-16)')

def load_harmonic_trajectories(n_traj, length=2048, seed=0):
    \"\"\"CONFIRM: paste verbatim from A1's OOD harmonic loader.\"\"\"
    raise NotImplementedError('Paste verbatim Harmonic generator from A1 OOD loader')

TRAJ_LOADERS = {
    'lorenz':   load_lorenz_trajectories,
    'rossler':  load_rossler_trajectories,
    'sprottb':  load_sprottb_trajectories,
    'burgers':  load_burgers_nu1_trajectories,
    'harmonic': load_harmonic_trajectories,
}
CLASS_NAMES = list(TRAJ_LOADERS.keys())
CHAOTIC_CLASSES = ['lorenz', 'rossler', 'sprottb']
NONCHAOTIC_CLASSES = ['burgers', 'harmonic']
print('Trajectory loaders registered (stubs -- fill in before running Section 6):', CLASS_NAMES)


## Section 6 — Feature extraction: Φ_pre and Φ_post per patch, per class

For each class: generate `N_FIT + N_HELDOUT` trajectories (60/class for eDMD budget),
run each through the model, collect per-patch (Φ_pre, Φ_post) pairs **in temporal order**
(needed for patch_t -> patch_t+1 pairing in eDMD), save to disk as `.npz` so the CPU-only
eDMD fitting step doesn't need the GPU/model loaded again.

Separately, `N_JACOBIAN` trajectories/class (15/class) are run with `requires_grad=True` for
the Jacobian-sensitivity section.

In [ ]:

N_FIT = 40
N_HELDOUT = 20
N_JACOBIAN = 15
TRAJ_LENGTH = 2048   # ADJUST: long enough to yield many patches per trajectory after context windowing
FEATURE_DIR = './a3_features'
os.makedirs(FEATURE_DIR, exist_ok=True)

def extract_patch_features(traj, model, device):
    \"\"\"Run one trajectory [C, T] through the model, return ordered arrays of
    Phi_pre [n_patches, d_pre] and Phi_post [n_patches, d_post] captured via the
    Section 3 hooks. Assumes the lift is applied once per forward call across all
    patches -- if the hook only fires once per call and returns a [n_patches, ...]
    tensor already, this is a direct passthrough; adjust if the model batches
    patches differently.\"\"\"
    x = torch.as_tensor(traj, dtype=torch.float32, device=device).unsqueeze(0)  # [1, C, T]
    with torch.no_grad():
        _ = model(x)
    pre = _captured['pre']
    post = _captured['post']
    pre_np = None if pre is None else pre.squeeze(0).cpu().numpy().reshape(-1, pre.shape[-1])
    post_np = post.squeeze(0).cpu().numpy().reshape(-1, post.shape[-1])
    return pre_np, post_np

for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    n_total = N_FIT + N_HELDOUT
    trajs = loader(n_total, length=TRAJ_LENGTH, seed=hash(cls) % (2**31))
    all_pre, all_post, traj_ids = [], [], []
    for i, traj in enumerate(trajs):
        pre_np, post_np = extract_patch_features(traj, model, device)
        all_post.append(post_np)
        traj_ids.append(np.full(post_np.shape[0], i))
        if pre_np is not None:
            all_pre.append(pre_np)
    post_arr = np.concatenate(all_post, axis=0)
    ids_arr = np.concatenate(traj_ids, axis=0)
    pre_arr = np.concatenate(all_pre, axis=0) if all_pre else None

    save_path = os.path.join(FEATURE_DIR, f'{cls}_features.npz')
    np.savez(save_path, phi_pre=pre_arr, phi_post=post_arr, traj_id=ids_arr,
             n_fit=N_FIT, n_heldout=N_HELDOUT)
    print(f'{cls}: saved {save_path}  '
          f'(post shape={post_arr.shape}, pre shape={None if pre_arr is None else pre_arr.shape}, '
          f'{n_total} trajectories)')

print('\nFeature extraction complete. GPU/model no longer needed for Sections 7-8.')


## Section 7 — eDMD fitting (CPU-only, numpy/scipy)

Shared ridge-regularized K, fit on a class-balanced pool across `N_FIT` trajectories/class,
scored on `N_HELDOUT` trajectories/class, repeated over `N_SPLITS` random fit/held-out
resamples. Run separately for Φ_pre and Φ_post — **do not merge the two tables.**

In [ ]:

import numpy as np
from scipy.stats import wilcoxon
from sklearn.linear_model import RidgeCV

N_SPLITS = 3
RIDGE_ALPHAS = np.logspace(-4, 3, 15)  # standard log-spaced grid for RidgeCV

def load_class_features(cls, which):  # which in {'phi_pre', 'phi_post'}
    d = np.load(os.path.join(FEATURE_DIR, f'{cls}_features.npz'), allow_pickle=True)
    feats = d[which]
    if feats.dtype == object or feats is None:
        return None, None
    return feats, d['traj_id']

def make_pairs(feats, traj_id):
    \"\"\"Consecutive-patch pairs (Phi_t, Phi_t+1) WITHIN each trajectory only
    (never pair the last patch of one trajectory with the first of the next).\"\"\"
    X, Y = [], []
    for tid in np.unique(traj_id):
        mask = traj_id == tid
        f = feats[mask]
        if f.shape[0] < 2:
            continue
        X.append(f[:-1])
        Y.append(f[1:])
    return np.concatenate(X, axis=0), np.concatenate(Y, axis=0)

def fit_and_score(which, seed):
    rng = np.random.default_rng(seed)
    fit_X, fit_Y, held = {}, {}, {}
    for cls in CLASS_NAMES:
        feats, traj_id = load_class_features(cls, which)
        if feats is None:
            print(f'[SKIP] {which} not available for class={cls}')
            return None
        traj_ids_unique = np.unique(traj_id)
        rng.shuffle(traj_ids_unique)
        fit_ids = set(traj_ids_unique[:N_FIT])
        held_ids = set(traj_ids_unique[N_FIT:N_FIT + N_HELDOUT])

        fit_mask = np.isin(traj_id, list(fit_ids))
        held_mask = np.isin(traj_id, list(held_ids))

        X_f, Y_f = make_pairs(feats[fit_mask], traj_id[fit_mask])
        fit_X[cls], fit_Y[cls] = X_f, Y_f
        held[cls] = (feats[held_mask], traj_id[held_mask])

    # pooled, class-balanced fit set: subsample every class down to the smallest
    # available #pairs so no class dominates
    min_pairs = min(fit_X[c].shape[0] for c in CLASS_NAMES)
    Xp, Yp = [], []
    for cls in CLASS_NAMES:
        idx = rng.choice(fit_X[cls].shape[0], size=min_pairs, replace=False)
        Xp.append(fit_X[cls][idx]); Yp.append(fit_Y[cls][idx])
    Xp = np.concatenate(Xp, axis=0); Yp = np.concatenate(Yp, axis=0)

    # RidgeCV per output dim is expensive; use a single shared alpha selected via
    # RidgeCV on a flattened multi-output fit (sklearn supports multi-output ridge).
    ridge = RidgeCV(alphas=RIDGE_ALPHAS, alpha_per_target=False)
    ridge.fit(Xp, Yp)
    K = ridge.coef_  # [d_out, d_in]
    intercept = ridge.intercept_
    print(f'  [{which}, seed={seed}] selected alpha={ridge.alpha_:.4g}, pooled pairs={Xp.shape[0]}')

    rows = []
    for cls in CLASS_NAMES:
        feats_h, traj_id_h = held[cls]
        Xh, Yh = make_pairs(feats_h, traj_id_h)
        pred = Xh @ K.T + intercept
        resid = np.linalg.norm(Yh - pred, axis=1) / (np.linalg.norm(Yh, axis=1) + 1e-12)
        rows.append({'class': cls, 'which': which, 'seed': seed,
                      'median_resid': float(np.median(resid)),
                      'iqr_low': float(np.percentile(resid, 25)),
                      'iqr_high': float(np.percentile(resid, 75)),
                      'n_pairs': len(resid), '_raw_resid': resid})
    return rows

all_rows = []
for which in ['phi_pre', 'phi_post']:
    for split_seed in range(N_SPLITS):
        rows = fit_and_score(which, seed=split_seed)
        if rows is not None:
            all_rows.extend(rows)

import pandas as pd
edmd_df = pd.DataFrame([{k: v for k, v in r.items() if k != '_raw_resid'} for r in all_rows])
print('\n=== eDMD residuals (median, IQR) by class / feature-space / split ===')
print(edmd_df.to_string(index=False))
edmd_df.to_csv('a3_edmd_residuals.csv', index=False)


In [ ]:

# Significance: chaotic classes vs. Burgers, paired Wilcoxon per split/feature-space,
# using the raw per-trajectory-pair residual distributions (not just the medians).
sig_rows = []
for which in ['phi_pre', 'phi_post']:
    for split_seed in range(N_SPLITS):
        subset = [r for r in all_rows if r['which'] == which and r['seed'] == split_seed]
        if not subset:
            continue
        burgers_resid = next((r['_raw_resid'] for r in subset if r['class'] == 'burgers'), None)
        if burgers_resid is None:
            continue
        for chaotic_cls in CHAOTIC_CLASSES:
            chaotic_resid = next((r['_raw_resid'] for r in subset if r['class'] == chaotic_cls), None)
            if chaotic_resid is None:
                continue
            n = min(len(burgers_resid), len(chaotic_resid))
            try:
                stat, p = wilcoxon(chaotic_resid[:n], burgers_resid[:n])
            except ValueError:
                p = float('nan')
            sig_rows.append({'which': which, 'seed': split_seed, 'class': chaotic_cls,
                              'compared_to': 'burgers',
                              'median_diff': float(np.median(chaotic_resid) - np.median(burgers_resid)),
                              'wilcoxon_p': p})

sig_df = pd.DataFrame(sig_rows)
print('=== Chaotic vs. Burgers residual comparison ===')
print(sig_df.to_string(index=False))
sig_df.to_csv('a3_edmd_significance.csv', index=False)
print('\n[REMINDER] positive median_diff = chaotic class has HIGHER (worse) residual than Burgers, '
      'i.e. consistent with the A1-motivated hypothesis. Check consistency ACROSS all 3 splits '
      'before treating any single split as confirmatory.')


## Section 8 — Jacobian sensitivity of the lift

Measures how much the lift itself amplifies small input perturbations, per class
(`N_JACOBIAN` trajectories/class, single pass — no CV needed, this is a direct measurement).
Uses `torch.autograd.functional.jacobian` (or a vector-Jacobian-product loop if the full
Jacobian is too large per patch) on the pre-projection dictionary Φ_pre w.r.t. the input
patch.

In [ ]:

def patch_lift_jacobian_norm(model, patch_tensor):
    \"\"\"patch_tensor: [patch_len] or [C, patch_len] single patch, requires_grad=True.
    Returns the Frobenius norm and spectral (largest singular value) norm of d(Phi_pre)/d(patch).
    CONFIRM: this assumes the lift can be called on a single patch in isolation -- if the repo's
    module signature requires the full [C, context_length] trajectory, wrap accordingly (run the
    full forward, but only backprop from one patch's Phi_pre output w.r.t. that patch's input
    slice of the trajectory -- a local Jacobian via autograd.grad, not the full-trajectory Jacobian).\"\"\"
    patch_tensor = patch_tensor.clone().requires_grad_(True)
    _ = model(patch_tensor.unsqueeze(0))
    phi = _captured['pre'] if _captured['pre'] is not None else _captured['post']
    phi = phi.reshape(-1)
    J = torch.zeros(phi.shape[0], patch_tensor.numel(), device=patch_tensor.device)
    for i in range(phi.shape[0]):
        grad_out = torch.zeros_like(phi)
        grad_out[i] = 1.0
        g, = torch.autograd.grad(phi, patch_tensor, grad_outputs=grad_out, retain_graph=True)
        J[i] = g.reshape(-1)
    fro_norm = torch.linalg.norm(J, ord='fro').item()
    spec_norm = torch.linalg.matrix_norm(J, ord=2).item()
    return fro_norm, spec_norm

jac_rows = []
for cls in CLASS_NAMES:
    loader = TRAJ_LOADERS[cls]
    trajs = loader(N_JACOBIAN, length=TRAJ_LENGTH, seed=(hash(cls) + 1) % (2**31))
    for traj in trajs:
        x = torch.as_tensor(traj, dtype=torch.float32, device=device)
        # Use only the first patch of this trajectory as a representative sample --
        # ADJUST to sample multiple patches per trajectory for a fuller distribution
        # if compute allows within the ~1-2 GPU-hour budget for this section.
        patch_length_local = getattr(model.config, 'patch_length', 16)
        patch = x[:, :patch_length_local]
        fro, spec = patch_lift_jacobian_norm(model, patch)
        jac_rows.append({'class': cls, 'fro_norm': fro, 'spectral_norm': spec})

jac_df = pd.DataFrame(jac_rows)
print(jac_df.groupby('class').agg(['median', lambda s: s.quantile(.25), lambda s: s.quantile(.75)]))
jac_df.to_csv('a3_jacobian_sensitivity.csv', index=False)


In [ ]:

# Significance for Jacobian norms, same chaotic-vs-Burgers comparison as Section 7
from scipy.stats import mannwhitneyu

jac_sig_rows = []
burgers_fro = jac_df[jac_df['class'] == 'burgers']['fro_norm'].values
for chaotic_cls in CHAOTIC_CLASSES:
    chaotic_fro = jac_df[jac_df['class'] == chaotic_cls]['fro_norm'].values
    if len(burgers_fro) and len(chaotic_fro):
        stat, p = mannwhitneyu(chaotic_fro, burgers_fro, alternative='two-sided')
        jac_sig_rows.append({'class': chaotic_cls, 'compared_to': 'burgers',
                              'median_diff': float(np.median(chaotic_fro) - np.median(burgers_fro)),
                              'mannwhitney_p': p})
jac_sig_df = pd.DataFrame(jac_sig_rows)
print(jac_sig_df.to_string(index=False))
jac_sig_df.to_csv('a3_jacobian_significance.csv', index=False)


## Section 9 — Diagnostic companions: effective rank + condition number

Cheap, forward-pass-only, descriptive context for Sections 7-8 — **not** used standalone as
evidence for/against the HYP (see chat discussion: rank/conditioning != linearizability).

In [ ]:

def effective_rank_and_condition(feats):
    # feats: [n_patches, d]
    feats_c = feats - feats.mean(axis=0, keepdims=True)
    cov = np.cov(feats_c, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.clip(eigvals, 1e-12, None)
    p = eigvals / eigvals.sum()
    effective_rank = np.exp(-np.sum(p * np.log(p)))  # exponential of entropy
    condition_number = eigvals.max() / eigvals.min()
    return effective_rank, condition_number

diag_rows = []
for which in ['phi_pre', 'phi_post']:
    for cls in CLASS_NAMES:
        feats, _ = load_class_features(cls, which)
        if feats is None:
            continue
        er, cond = effective_rank_and_condition(feats)
        diag_rows.append({'which': which, 'class': cls, 'effective_rank': er,
                           'condition_number': cond, 'ambient_dim': feats.shape[1]})

diag_df = pd.DataFrame(diag_rows)
print(diag_df.to_string(index=False))
diag_df.to_csv('a3_geometry_diagnostics.csv', index=False)


## Section 10 — Summary table + reading against the pre-registered map

Pulls everything into one table and prints the interpretation cell (from Section 0) that
matches the observed pattern — **read this cell's printed output, don't just eyeball the raw
tables**, since the whole point of pre-registering the map was to avoid post-hoc narrative
fitting.

In [ ]:

print('=== A3 SUMMARY ===\n')
print('--- eDMD median residual by class (median across 3 splits) ---')
print(edmd_df.groupby(['which', 'class'])['median_resid'].median().unstack('which'))

print('\n--- eDMD significance (chaotic vs. Burgers), fraction of 3 splits with p<0.05 ---')
if len(sig_df):
    sig_summary = sig_df.groupby(['which', 'class']).apply(
        lambda g: (g['wilcoxon_p'] < 0.05).mean()
    )
    print(sig_summary)

print('\n--- Jacobian Frobenius norm by class (median) ---')
print(jac_df.groupby('class')['fro_norm'].median())

print('\n--- Geometry diagnostics (context only, not standalone evidence) ---')
print(diag_df)

print(\"\"\"
[REMINDER -- read against Section 0's pre-registered interpretation map]
- Chaotic classes (lorenz/rossler/sprottb) showing consistently HIGHER eDMD residual than
  Burgers, replicated across the 3 splits, in BOTH phi_pre and phi_post -> clean mechanistic
  confirmation of A1's pattern.
- Same in phi_pre only, NOT phi_post -> the trained projection is compensating downstream;
  A1's behavioral split needs a different/additional explanation.
- No consistent separation anywhere -> negative result; push to A2a (temporal attention).
- Report at MEDIUM confidence regardless of outcome (see budget/limitations in Section 0) --
  this is a first-pass test, not a definitive mechanism claim.
\"\"\")
